In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_df = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv')

# Display the first few rows of the dataframe
print(train_df.head())

# Check the data types of each column
print(train_df.dtypes)

# Summary statistics for numerical columns
print(train_df.describe())

# Check for missing values
print(train_df.isnull().sum())

# Separate numerical and categorical columns
numerical_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(exclude=[np.number]).columns

# Distribution of numerical features
for col in numerical_cols:
    plt.figure(figsize=(10, 4))
    sns.histplot(train_df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

# Distribution of categorical features
for col in categorical_cols:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=train_df, x=col)
    plt.title(f'Distribution of {col}')
    plt.show()

# Correlation matrix for numerical features
plt.figure(figsize=(12, 8))
sns.heatmap(train_df[numerical_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


  surgery  hospital_number  ...  capillary_refill_time     outcome
0     yes           527706  ...             less_3_sec        died
1     yes           528641  ...             less_3_sec       lived
2     yes           535043  ...             more_3_sec  euthanized
3     yes           535043  ...             less_3_sec  euthanized
4     yes           528890  ...             more_3_sec        died

[5 rows x 9 columns]
surgery                   object
hospital_number            int64
rectal_temp              float64
pulse                    float64
respiratory_rate         float64
peripheral_pulse          object
mucous_membrane           object
capillary_refill_time     object
outcome                   object
dtype: object
       hospital_number  rectal_temp       pulse  respiratory_rate
count     9.860000e+02   986.000000  986.000000        986.000000
mean      9.746363e+05    38.186207   79.477688         30.056795
std       1.385090e+06     0.774303   29.168581         16.318555
m

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-30 21:22:28.715 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'outcome'], 'Numeric': ['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Copy the original DataFrame to avoid modifying the original data
train_df_copy = train_df.copy()

# Handling missing values
# For numerical features, use the median to fill missing values
num_features = ['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate']
fill_missing_num = FillMissingValue(features=num_features, strategy='median')
train_df_copy = fill_missing_num.fit_transform(train_df_copy)

# For categorical features, use the most frequent value to fill missing values
cat_features = ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time']
fill_missing_cat = FillMissingValue(features=cat_features, strategy='most_frequent')
train_df_copy = fill_missing_cat.fit_transform(train_df_copy)

# Encoding categorical variables
# Use label encoding for categorical features
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    train_df_copy[col] = le.fit_transform(train_df_copy[col])
    label_encoders[col] = le

# Normalizing numerical features
scaler = StandardScale(features=num_features)
train_df_copy = scaler.fit_transform(train_df_copy)

# Display the preprocessed DataFrame
print(train_df_copy.head())


   surgery  hospital_number  ...  capillary_refill_time     outcome
0        1        -0.322836  ...                      0        died
1        1        -0.322161  ...                      0       lived
2        1        -0.317536  ...                      1  euthanized
3        1        -0.317536  ...                      0  euthanized
4        1        -0.321981  ...                      1        died

[5 rows x 9 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': ['outcome'], 'Numeric': ['surgery', 'hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from metagpt.tools.libs.data_preprocess import LabelEncode

# Label encode the outcome column
label_encode_outcome = LabelEncode(features=['outcome'])
train_df_copy = label_encode_outcome.fit_transform(train_df_copy)

# Split the data into features and target
X = train_df_copy.drop(columns=['outcome'])
y = train_df_copy['outcome']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the XGBoost classifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Predict on the test set
y_pred = best_model.predict(X_test)

# Calculate the F1 score
f1 = f1_score(y_test, y_pred, average='macro')
print(f'F1 Score on the test set: {f1}')


F1 Score on the test set: 0.606986912869266


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [21:23:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
